In [ ]:
# v44 — Euclid P1: Mirror Correlation Test
# Pre-registered prediction: peak at r = 830 Mpc (T1/2)
#
# KTF3 identification phi(x,y,z) = (-x,y,z):
# Galaxy at position (x,y,z) is topologically connected to (-x,y,z).
# Cross-correlation C(r, phi(r)) should peak at r = T1/2 = 830 Mpc.
#
# PRE-REGISTERED: March 2026, before Euclid DR1 release.
# Run when Euclid DR1 data is available (expected October 2026).
#
# Cotting, S. (2026)
# academia.edu/AndrewCotting

In [ ]:
# CELL 1 -- Imports
!pip install numpy matplotlib scipy astropy -q
import numpy as np
import matplotlib.pyplot as plt
from astropy.cosmology import FlatLambdaCDM
from astropy.io import fits
import os, warnings
warnings.filterwarnings('ignore')

cosmo = FlatLambdaCDM(H0=67.4, Om0=0.315)

print('v44 -- Euclid P1: Mirror Correlation Test')
print('PRE-REGISTERED March 2026 -- Cotting, S.')
print('='*60)
print()
print('KTF3 Prediction P1:')
print('  Cross-correlation C(r, phi(r)) peaks at r = T1/2 = 830 Mpc')
print('  phi(x,y,z) = (-x, y, z)  [Klein-bottle identification]')
print()
print('Falsified if: no peak at r = 830 Mpc')
print('Confirmed if: peak at r = 830 +/- 100 Mpc, sigma > 2')

In [ ]:
# CELL 2 -- Load Euclid DR1 galaxy catalog
# Replace this path with actual Euclid DR1 file when available
# Expected format: FITS file with columns RA, DEC, Z (or ZPHOT)

EUCLID_FILE = 'euclid_dr1_galaxies.fits'  # << REPLACE WITH ACTUAL FILE

if not os.path.exists(EUCLID_FILE):
    print('Euclid DR1 not yet available.')
    print('Generating synthetic test catalog to validate pipeline...')
    print()
    
    # Synthetic Euclid-like catalog
    N_gal = 100000
    rng = np.random.default_rng(42)
    
    # Euclid footprint: ~15000 deg^2, z=0.1-2.0
    ra  = rng.uniform(0, 360, N_gal)
    dec = np.degrees(np.arcsin(rng.uniform(-0.5, 0.9, N_gal)))
    z   = rng.uniform(0.1, 2.0, N_gal)
    
    print(f'Synthetic galaxies: {N_gal}')
    USE_SYNTHETIC = True
else:
    print(f'Loading {EUCLID_FILE}...')
    hdul = fits.open(EUCLID_FILE)
    data = hdul[1].data
    ra   = data['RA']
    dec  = data['DEC']
    z    = data['Z'] if 'Z' in data.dtype.names else data['ZPHOT']
    print(f'Galaxies loaded: {len(ra)}')
    USE_SYNTHETIC = False

# Convert to comoving Cartesian
print('Converting to comoving coordinates...')
chi = cosmo.comoving_distance(z).value  # Mpc
ra_rad  = np.radians(ra)
dec_rad = np.radians(dec)
x = chi * np.cos(dec_rad) * np.cos(ra_rad)
y = chi * np.cos(dec_rad) * np.sin(ra_rad)
z_cart = chi * np.sin(dec_rad)
pos = np.stack([x, y, z_cart], axis=1)

# Topological mirror: phi(x,y,z) = (-x, y, z)
pos_mirror = np.stack([-x, y, z_cart], axis=1)

print(f'Position range x: {x.min():.0f} to {x.max():.0f} Mpc')
print(f'Position range y: {y.min():.0f} to {y.max():.0f} Mpc')
print(f'Survey volume: ~{len(ra)/1e6:.1f}M galaxies')

In [ ]:
# CELL 3 -- Compute mirror cross-correlation C(r, phi(r))
# For each galaxy pair (i,j): compute separation r and
# check if galaxy j is near the mirror position phi(galaxy_i)
# C_mirror(r) = fraction of galaxy pairs at separation r
# where one galaxy is near the mirror of the other

T1     = 1660   # Mpc
r_pred = T1 / 2  # = 830 Mpc -- predicted peak

print(f'KTF3 prediction: mirror correlation peak at r = {r_pred} Mpc')
print(f'Testing r range: 200 - 2000 Mpc')
print()

# Separation bins
bins = np.arange(200, 2001, 100)  # 100 Mpc bins
bin_centers = 0.5 * (bins[:-1] + bins[1:])
N_BINS = len(bin_centers)

# Sample N_PAIRS random galaxy pairs
N_PAIRS = 500000
rng = np.random.default_rng(42)
N_gal = len(pos)
idx1 = rng.integers(0, N_gal, N_PAIRS)
idx2 = rng.integers(0, N_gal, N_PAIRS)
keep = idx1 != idx2
idx1, idx2 = idx1[keep], idx2[keep]

# Standard separation r(i,j)
dr_std    = pos[idx2] - pos[idx1]
r_std     = np.linalg.norm(dr_std, axis=1)

# Mirror separation: distance from galaxy j to mirror of galaxy i
dr_mirror = pos[idx2] - pos_mirror[idx1]
r_mirror  = np.linalg.norm(dr_mirror, axis=1)

# Compute C_mirror(r) - C_std(r) per bin
# Using galaxy density as proxy: count pairs in each bin
N_std    = np.zeros(N_BINS)
N_mirror = np.zeros(N_BINS)

for b in range(N_BINS):
    N_std[b]    = ((r_std    >= bins[b]) & (r_std    < bins[b+1])).sum()
    N_mirror[b] = ((r_mirror >= bins[b]) & (r_mirror < bins[b+1])).sum()

# Normalize
N_total = len(idx1)
C_std    = N_std    / N_total
C_mirror = N_mirror / N_total
C_excess = C_mirror - C_std

# Peak location
peak_bin = np.argmax(C_excess)
peak_r   = bin_centers[peak_bin]

print(f'Mirror excess peak at: r = {peak_r:.0f} Mpc')
print(f'Predicted peak at:     r = {r_pred:.0f} Mpc')
print(f'Offset: {abs(peak_r - r_pred):.0f} Mpc')

In [ ]:
# CELL 4 -- Monte Carlo significance
N_MC = 300
print(f'Monte Carlo null (N={N_MC}): randomising mirror axis...')

mc_excess = np.zeros((N_MC, N_BINS))
rng_mc = np.random.default_rng(99)

for i in range(N_MC):
    # Random rotation of mirror axis (null: no preferred axis)
    theta_rand = rng_mc.uniform(0, np.pi)
    phi_rand   = rng_mc.uniform(0, 2*np.pi)
    
    # Random mirror direction
    n = np.array([np.sin(theta_rand)*np.cos(phi_rand),
                  np.sin(theta_rand)*np.sin(phi_rand),
                  np.cos(theta_rand)])
    
    # Mirror reflection in random direction
    pos_rand_mirror = pos - 2 * np.outer(pos @ n, n)
    
    dr_m = pos_rand_mirror[idx1] - pos[idx2]
    r_m  = np.linalg.norm(dr_m, axis=1)
    
    N_m = np.zeros(N_BINS)
    for b in range(N_BINS):
        N_m[b] = ((r_m >= bins[b]) & (r_m < bins[b+1])).sum()
    mc_excess[i] = N_m/N_total - C_std
    
    if (i+1) % 75 == 0:
        print(f'  MC {i+1}/{N_MC}')

mc_mean = mc_excess.mean(axis=0)
mc_std  = mc_excess.std(axis=0)
sigma   = (C_excess - mc_mean) / (mc_std + 1e-10)

# Sigma at predicted peak
pred_bin  = np.argmin(np.abs(bin_centers - r_pred))
sigma_pred = sigma[pred_bin]

print(f'\nSigma at r = {r_pred} Mpc: sigma = {sigma_pred:.2f}')
print(f'Max sigma overall: {sigma.max():.2f} at r = {bin_centers[sigma.argmax()]:.0f} Mpc')

In [ ]:
# CELL 5 -- Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')
for ax in axes:
    ax.set_facecolor('#0d1117'); ax.tick_params(colors='white'); ax.spines[:].set_color('#444')

ax = axes[0]
ax.fill_between(bin_centers, mc_mean-2*mc_std, mc_mean+2*mc_std,
                alpha=0.3, color='gray', label='2sigma null')
ax.plot(bin_centers, C_excess, color='#EF5350', lw=2, label='Mirror excess')
ax.axvline(r_pred, color='yellow', lw=2, ls='--', label=f'T1/2={r_pred} Mpc (predicted)')
ax.axhline(0, color='white', lw=0.5, alpha=0.4)
ax.set_xlabel('r [Mpc]', color='white'); ax.set_ylabel('C_mirror - C_std', color='white')
ax.set_title('P1: Mirror Correlation Excess\n(KTF3: peak at 830 Mpc)', color='white')
ax.legend(facecolor='#1a1a2e', labelcolor='white')

ax = axes[1]
ax.plot(bin_centers, sigma, color='#EF5350', lw=2)
ax.axhline(0,  color='white',  lw=0.5, alpha=0.4)
ax.axhline(2,  color='orange', lw=1.5, ls='--', alpha=0.7, label='2sigma')
ax.axhline(-2, color='orange', lw=1.5, ls='--', alpha=0.7)
ax.axvline(r_pred, color='yellow', lw=2, ls='--', label=f'Predicted: {r_pred} Mpc')
ax.set_xlabel('r [Mpc]', color='white'); ax.set_ylabel('Significance sigma', color='white')
ax.set_title(f'P1 Significance\nsigma at 830 Mpc = {sigma_pred:.2f}', color='white')
ax.legend(facecolor='#1a1a2e', labelcolor='white')

plt.suptitle('v44: Euclid P1 Mirror Correlation -- Cotting (2026)', color='white', fontsize=13)
plt.tight_layout()
plt.savefig('v44_euclid_p1.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# CELL 6 -- Summary
print('\n' + '='*60)
print('RESULT v44: EUCLID P1 MIRROR CORRELATION')
print('Cotting, S. (2026) -- PRE-REGISTERED March 2026')
print('='*60)
print(f'  Catalog: {"SYNTHETIC (pipeline test)" if USE_SYNTHETIC else EUCLID_FILE}')
print(f'  Galaxies: {N_gal}')
print(f'  Pairs sampled: {len(idx1)}')
print()
print(f'  Predicted peak: r = {r_pred} Mpc')
print(f'  Observed peak:  r = {peak_r:.0f} Mpc')
print(f'  Sigma at r=830: {sigma_pred:.2f}')
print(f'  Max sigma:      {sigma.max():.2f} at r={bin_centers[sigma.argmax()]:.0f} Mpc')
print()
if sigma_pred > 2:
    verdict = 'CONFIRMED -- Peak at predicted location. Consistent with KTF3.'
elif sigma_pred > 1:
    verdict = 'MARGINAL -- Weak excess at predicted location.'
elif sigma_pred > 0:
    verdict = 'WEAK -- Correct sign but not significant.'
else:
    verdict = 'NULL/FALSIFIED -- No mirror excess at predicted location.'
print(f'  Verdict: {verdict}')
print()
print('  Pre-registration: academia.edu/AndrewCotting')
print('  Contact: andrew.cotting@gmail.com')
print('='*60)